# 🧪 SapBERT Thai — ทดสอบโมเดลของเรา 3 รูปแบบ

ทดสอบโมเดล **SapBERT Thai Extended** ของเรา ด้วย 3 รูปแบบการทดสอบ  
เทียบกับแนวทางของ [WangchanBERTa (AIResearch.in.th)](https://airesearch.in.th/releases/wangchanberta-pre-trained-thai-language-model/)

| # | การทดสอบ | คำอธิบาย |
|:--|:---|:---|
| 1 | **Tokenization Benchmark** | เปรียบเทียบการตัดคำ 4 ระดับ |
| 2 | **MLM (fill-mask)** | ทำนายคำที่ถูกปิดบัง ทดสอบความรู้ทางการแพทย์ |
| 3 | **Token Classification (NER)** | จำแนก DRUG / DISEASE / SYMPTOM |


## ⚙️ Setup & ติดตั้ง Packages

In [1]:
import os
import sys
import subprocess
from pathlib import Path

# ตั้งค่า Root directory ของโปรเจกต์
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project Root:', project_root)

# ติดตั้ง packages ที่จำเป็น
required = ['transformers', 'pythainlp', 'torch', 'sentencepiece', 'peft']
for pkg in required:
    try:
        __import__(pkg)
        print('OK:', pkg)
    except ImportError:
        print('Installing:', pkg)
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print('Done:', pkg)

Project Root: i:\NECTEC\thai_bert_lora


i:\NECTEC\thai_bert_lora\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: transformers
OK: pythainlp
OK: torch
OK: sentencepiece
OK: peft


In [2]:
from transformers import AutoTokenizer

# ค้นหา Path โมเดล SapBERT Thai ของเรา (ลำดับความสำคัญ: merged > extended > raw)
merged_path   = project_root / 'outputs' / 'merged_model_1'
extended_path = project_root / 'data' / 'interim' / 'extended_base'
raw_path      = project_root / 'data' / 'interim' / 'sapbert_raw'

if merged_path.exists():
    base_model_path = str(merged_path)
    print('Using merged model (LoRA merged):', base_model_path)
elif extended_path.exists():
    base_model_path = str(extended_path)
    print('Using extended base model:', base_model_path)
elif raw_path.exists():
    base_model_path = str(raw_path)
    print('Using raw SapBERT model:', base_model_path)
else:
    base_model_path = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
    print('Using original SapBERT from HuggingFace (โมเดลของเราไม่พบ):', base_model_path)

tokenizer = AutoTokenizer.from_pretrained(base_model_path, use_fast=False)
print('Vocab size:', len(tokenizer), 'tokens')

Using merged model (LoRA merged): i:\NECTEC\thai_bert_lora\outputs\merged_model_1
Vocab size: 40075 tokens


---
## 📊 ส่วนที่ 1: Tokenization Benchmark
เปรียบเทียบการตัดคำ 4 ระดับ: Word / Subword (BERT ของเรา) / TCC / Character

In [3]:
from pythainlp.tokenize import word_tokenize, subword_tokenize

def analyze_tokenization(text):
    sep = ' | '
    print('=' * 75)
    print(' Text:', text)
    print('=' * 75)

    words = word_tokenize(text, engine='newmm')
    print('1. หน่วยคำ (Word -', len(words), 'คำ)          :', sep.join(words))

    subwords = tokenizer.tokenize(text)
    print('2. หน่วยคำย่อย (BERT Subword -', len(subwords), 'tokens):', sep.join(subwords))

    tcc_units = subword_tokenize(text, engine='tcc')
    print('3. หน่วยพยางค์/TCC (', len(tcc_units), 'units)    :', sep.join(tcc_units))

    chars = list(text)
    print('4. หน่วยตัวอักษร (Char -', len(chars), 'ตัว)   :', sep.join(chars))
    print()

test_sentences = [
    'วันนี้ฉันสั่งกิวดังมาทานที่บ้าน',
    'อะบิราเทอโรน ใช้รักษามะเร็งต่อมลูกหมาก โดยมีอาการแพร่กระจาย',
    'อะดาลิมูแมบ ใช้รักษาข้ออักเสบรูมาตอยด์และโรคโครห์น โดยมีอาการอักเสบ',
    'พาราเซตามอล บรรเทาอาการปวดและลดไข้',
]

for sentence in test_sentences:
    analyze_tokenization(sentence)

 Text: วันนี้ฉันสั่งกิวดังมาทานที่บ้าน
1. หน่วยคำ (Word - 9 คำ)          : วันนี้ | ฉัน | สั่ง | กิว | ดัง | มา | ทาน | ที่ | บ้าน
2. หน่วยคำย่อย (BERT Subword - 11 tokens): วัน | ##นี้ | ##ฉัน | ##สั่ง | ##กิ | ##วด | ##ัง | ##มา | ##ทาน | ##ที่ | ##บ้าน
3. หน่วยพยางค์/TCC ( 16 units)    : วั | น | นี้ | ฉั | น | สั่ง | กิ | ว | ดั | ง | มา | ทา | น | ที่ | บ้า | น
4. หน่วยตัวอักษร (Char - 31 ตัว)   : ว | ั | น | น | ี | ้ | ฉ | ั | น | ส | ั | ่ | ง | ก | ิ | ว | ด | ั | ง | ม | า | ท | า | น | ท | ี | ่ | บ | ้ | า | น

 Text: อะบิราเทอโรน ใช้รักษามะเร็งต่อมลูกหมาก โดยมีอาการแพร่กระจาย
1. หน่วยคำ (Word - 17 คำ)          : อะ | บิ | รา | เท | อ | โร | น |   | ใช้ | รักษา | มะเร็ง | ต่อมลูกหมาก |   | โดย | มี | อาการ | แพร่กระจาย
2. หน่วยคำย่อย (BERT Subword - 20 tokens): อะ | ##บิ | ##รา | ##เท | ##อ | ##โร | ##น | ใช้ | ##รักษา | ##มะ | ##เร็ง | ##ต่อ | ##มล | ##ูก | ##หม | ##าก | โดยมี | ##อาการ | ##แพร่ | ##กระจาย
3. หน่วยพยางค์/TCC ( 35 units)    : อะ | บิ | รา | เท | อ | โร | น 

---
## 🔮 ส่วนที่ 2: MLM fill-mask — ทำนายคำที่ถูกปิดบัง
ทดสอบว่าโมเดลของเรามีความเข้าใจคำศัพท์ทางการแพทย์ภาษาไทยมากน้อยแค่ไหน  
โดยให้โมเดลทำนายคำที่ถูกปิดบัง (`[MASK]`) ในประโยค

In [4]:
from transformers import AutoModelForMaskedLM, pipeline

print('Loading SapBERT Thai model for fill-mask...')
mlm_model = AutoModelForMaskedLM.from_pretrained(base_model_path)

fill_mask = pipeline(
    task='fill-mask',
    tokenizer=tokenizer,
    model=mlm_model
)
print('Done! Mask token:', tokenizer.mask_token)

Loading SapBERT Thai model for fill-mask...


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 1253.23it/s]
[transformers] BertForMaskedLM LOAD REPORT from: i:\NECTEC\thai_bert_lora\outputs\merged_model_1
Key                          | Status  | 
-----------------------------+---------+-
cls.predictions.decoder.bias | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Done! Mask token: [MASK]


In [14]:
mask = tokenizer.mask_token  # [MASK]

mlm_tests = [
    f'เวลาฝนตก ถนนมักจะ{mask}และเดินทางลำบาก',

    f'ฉันกำลังเดินทางไป{mask}เพื่อซื้อของใช้ในบ้าน',

    f'หลังเลิกเรียน นักเรียนเดินทางกลับ{mask}',

    f'พรุ่งนี้ฉันต้อง{mask}แต่เช้าเพื่อไปประชุม',
]

print('=== MLM fill-mask Results (SapBERT Thai) ===')
for text in mlm_tests:
    print('Input :', text)
    try:
        results = fill_mask(text, top_k=3)
        print('Top-3 predictions:')
        for r in results:
            print('  [', r['token_str'], ']  score:', round(r['score'], 4))
    except Exception as e:
        print('  Error:', e)
    print()

=== MLM fill-mask Results (SapBERT Thai) ===
Input : เวลาฝนตก ถนนมักจะ[MASK]และเดินทางลำบาก
Top-3 predictions:
  [ ##เดินทาง ]  score: 0.1046
  [ ##วิ่ง ]  score: 0.0543
  [ ##บิน ]  score: 0.0345

Input : ฉันกำลังเดินทางไป[MASK]เพื่อซื้อของใช้ในบ้าน
Top-3 predictions:
  [ ##บ้าน ]  score: 0.0564
  [ ##ไหน ]  score: 0.0287
  [ ##แล้ว ]  score: 0.027

Input : หลังเลิกเรียน นักเรียนเดินทางกลับ[MASK]
Top-3 predictions:
  [ ##โรงเรียน ]  score: 0.1283
  [ ##ประเทศไทย ]  score: 0.1124
  [ ##บ้าน ]  score: 0.1081

Input : พรุ่งนี้ฉันต้อง[MASK]แต่เช้าเพื่อไปประชุม
Top-3 predictions:
  [ ##เรียน ]  score: 0.0357
  [ ##ตัดสิน ]  score: 0.0272
  [ ##แพ้ ]  score: 0.0268



---
## 🏷️ ส่วนที่ 3: Token Classification (NER) — DRUG / DISEASE / SYMPTOM
ทดสอบโมเดล NER ที่ fine-tune มาจาก SapBERT Thai ของเรา  
จำแนก entity 3 ประเภท: **DRUG** (ยา), **DISEASE** (โรค), **SYMPTOM** (อาการ)

In [17]:
from transformers import AutoModelForTokenClassification
import json

# ค้นหา NER model ที่ fine-tune แล้ว
ner_candidates = [
    project_root / 'outputs' / 'ner_model',
    project_root / 'outputs' / 'ner_lora_model',
]

ner_model_path = None
for p in ner_candidates:
    if p.exists():
        ner_model_path = str(p)
        break

if ner_model_path:
    print('Found NER model:', ner_model_path)
    ner_tokenizer = AutoTokenizer.from_pretrained(ner_model_path, use_fast=False)
    ner_model = AutoModelForTokenClassification.from_pretrained(ner_model_path)
    ner_pipeline = pipeline(
        task='ner',
        tokenizer=ner_tokenizer,
        model=ner_model,
        aggregation_strategy='simple'
    )
    print('NER model loaded! Labels:', list(ner_model.config.id2label.values()))
else:
    print('NER model not found — กรุณา run train_sapbert_ner.py ก่อน')
    print('Paths checked:')
    for p in ner_candidates:
        print(' ', p)
    ner_pipeline = None

NER model not found — กรุณา run train_sapbert_ner.py ก่อน
Paths checked:
  i:\NECTEC\thai_bert_lora\outputs\ner_model
  i:\NECTEC\thai_bert_lora\outputs\ner_lora_model


In [20]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

NER_MODEL_PATH = r'I:\NECTEC\thai_bert_lora\outputs\merged_model_1_ner_base'

print('Loading NER model from:', NER_MODEL_PATH)
ner_tok = AutoTokenizer.from_pretrained(NER_MODEL_PATH, use_fast=False)
ner_model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_PATH)
ner_pipe = pipeline(
    task='ner',
    tokenizer=ner_tok,
    model=ner_model,
    aggregation_strategy='simple'
)
print('Labels:', list(ner_model.config.id2label.values()))

emoji = {'DRUG': '', 'DISEASE': '', 'SYMPTOM': ''}

def show_ner(text):
    entities = ner_pipe(text)
    if entities:
        for ent in entities:
            label = ent.get('entity_group', ent.get('entity', '?'))
            print(f"  {emoji.get(label,'🔖')} {label:8s}: \"{ent['word']}\"  ({ent['score']:.4f})")
    else:
        print('  (ไม่พบ entity)')
    print()

# ─────────────────────────────────────────────────────────────────
# ส่วนที่ 1: ประโยคที่อยู่ใน Training Dataset (In-distribution)
# ─────────────────────────────────────────────────────────────────
print('=' * 65)
print('📚 ส่วนที่ 1 — ประโยคที่ใช้ฝึกโมเดล (In-distribution)')
print('=' * 65)

in_dist = [
    # Text_TH_003 — อะบิราเทอโรน
    'อะบิราเทอโรน ใช้รักษา มะเร็งต่อมลูกหมาก โดยมีอาการ แพร่กระจาย',
    # Text_TH_007 — อะดาลิมูแมบ
    'อะดาลิมูแมบ ใช้รักษา ข้ออักเสบรูมาตอยด์ โดยมีอาการ อักเสบและแผลเรื้อรัง',
    # Text_TH_018 — อัลโลพูรินอล
    'อัลโลพูรินอล ใช้รักษา โรคเกาต์และนิ่วในไต',
    # Text_TH_031 — คลอร์เฟนิรามีน
    'คลอร์เฟนิรามีน มาลีเอต ใช้รักษา โรคหวัด โดยมีอาการ น้ำมูกไหลและตาแดงและจาม',
]

for text in in_dist:
    print('Text:', text)
    show_ner(text)

# ─────────────────────────────────────────────────────────────────
# ส่วนที่ 2: ประโยคนอก Dataset (Out-of-distribution)
# ─────────────────────────────────────────────────────────────────
print('=' * 65)
print('🌐 ส่วนที่ 2 — ประโยคนอก Dataset (Out-of-distribution)')
print('=' * 65)

out_dist = [
    # ยาทั่วไปที่ไม่อยู่ใน dataset
    'แพทย์สั่งยาพาราเซตามอลและไอบูโพรเฟนให้ผู้ป่วยที่มีไข้สูงและปวดศีรษะ',
    # โรคที่ไม่อยู่ใน dataset
    'ผู้ป่วยโรคเบาหวานชนิดที่สองมีอาการกระหายน้ำและปัสสาวะบ่อยผิดปกติ',
    # ประโยคข่าวสุขภาพทั่วไป
    'กรมอนามัยแนะนำให้ฉีดวัคซีนไข้หวัดใหญ่ทุกปีเพื่อป้องกันการติดเชื้อ',
    # ประโยคบทสนทนาสูตรธรรมชาติ
    'หมอบอกว่าฉันเป็นโรคความดันโลหิตสูง ต้องกินยาอะมโลดิพีนทุกวัน',
]

for text in out_dist:
    print('Text:', text)
    show_ner(text)


Loading NER model from: I:\NECTEC\thai_bert_lora\outputs\merged_model_1_ner_base


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6841.14it/s]

Labels:

 ['O', 'B-DRUG', 'I-DRUG', 'B-DISEASE', 'I-DISEASE', 'B-SYMPTOM', 'I-SYMPTOM']
📚 ส่วนที่ 1 — ประโยคที่ใช้ฝึกโมเดล (In-distribution)
Text: อะบิราเทอโรน ใช้รักษา มะเร็งต่อมลูกหมาก โดยมีอาการ แพร่กระจาย
   DRUG    : "อะบิราเทอโรน"  (0.9998)
   DISEASE : "มะเร็งต่อมลูกหมาก"  (0.9998)
   SYMPTOM : "แพร่กระจาย"  (0.9996)

Text: อะดาลิมูแมบ ใช้รักษา ข้ออักเสบรูมาตอยด์ โดยมีอาการ อักเสบและแผลเรื้อรัง
   DRUG    : "อะดาลิมูแมบ"  (0.9998)
   DISEASE : "ข้ออักเสบรูมาตอยด์"  (0.9994)
   SYMPTOM : "อักเสบ"  (0.9998)
   SYMPTOM : "##แผลเรื้อรัง"  (0.9998)

Text: อัลโลพูรินอล ใช้รักษา โรคเกาต์และนิ่วในไต
   DRUG    : "อัลโลพูรินอล"  (0.9997)
   DISEASE : "โรคเกาต์"  (0.9998)
   DISEASE : "##นิ่วในไต"  (0.9996)

Text: คลอร์เฟนิรามีน มาลีเอต ใช้รักษา โรคหวัด โดยมีอาการ น้ำมูกไหลและตาแดงและจาม
   DRUG    : "คลอร์เฟนิรามีน มาลีเอต"  (0.9998)
   DISEASE : "โรคหวัด"  (0.9996)
   SYMPTOM : "น้ำมูกไหล"  (0.9998)
   SYMPTOM : "##ตาแดง"  (0.9996)
   SYMPTOM : "##จาม"  (0.9996)

🌐 ส่วนที่ 2 — ประโยคนอก Dataset 